# Create a Schema for the data generator (patient vitals)

In [0]:
%sql
-- This creates the catalog only if it doesn't already exist
CREATE CATALOG IF NOT EXISTS patient_data;

-- Optional: Add a comment for metadata clarity
COMMENT ON CATALOG patient_data IS 'The "Patient" data catalog';

# Create a Volume Path (Data Generator) (patient vitals)

In [0]:
import json
import time
import random
import os
from datetime import datetime

# 1. Variables
catalog_name = "patient_data"
schema_name = "ingestion_patient_vitals"
volume_name = "raw_jsons"

# 2. Create the Catalog if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

# 3. Create the Schema (Database) inside that catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# 4. Create the Volume inside the schema
# We use 'CREATE VOLUME' for a Managed Volume (easiest setup)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

# 5. Now you can use dbutils to create sub-folders inside that volume
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/"
dbutils.fs.mkdirs(volume_path)

print(f"Success! Path is ready: {volume_path}")

# Create the Bronze, Silver and Gold layer (patient vitals)

In [0]:
import json
import time
import random
import os
from datetime import datetime

# 1. Variables
catalog_name = "patient_data"

# 2. Create the Bronze Schema (Database) inside that catalog
schema_name = "bronze_patient_vitals"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# 3. Create the Silver Schema (Database) inside that catalog
schema_name = "silver_patient_vitals"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# 4. Create the Silver Schema (Database) inside that catalog
schema_name = "gold_patient_vitals"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# Create the cdc and silver scd type 2 schemas (Patient Profiles)


In [0]:
import json
import time
import random
import os
from datetime import datetime

# 1. Variables
catalog_name = "patient_data"

# 2. Create the Bronze Schema (Database) inside that catalog
schema_name = "bronze_patients"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# 2. Create the Bronze Schema (Database) inside that catalog
schema_name = "silver_patients"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# Insert data into the bronze cdc table (Patient Profiles)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from datetime import datetime, timedelta
import random

# Initialize Spark
spark = SparkSession.builder.appName("CDC_Bronze_Simulation").getOrCreate()

# 1. Define your schema
schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("height_cm", LongType(), True),  
    StructField("age", LongType(), True),        
    StructField("sex", StringType(), True),
    StructField("op", StringType(), True), # CDC Operation: I, U, D
    StructField("updated_at", TimestampType(), True)
])

# 2. Simulated CDC Data based on your image
# We'll simulate a mix of Inserts (I) and Updates (U)
base_time = datetime.now()

data = [
    ("PT-1",   "Dave Silva",  85.2, 182, 42, "M", "I", base_time - timedelta(hours=5)),
    ("PT-158", "Jane Smith",  62.5, 160, 31, "F", "I", base_time - timedelta(hours=4)),
    ("PT-2",   "Jane Smith",  70.1, 168, 28, "F", "I", base_time - timedelta(hours=3)),
    ("PT-3",   "John Doe",    90.0, 185, 50, "M", "I", base_time - timedelta(hours=2)),
    ("PT-318", "Bob Miller",  78.4, 175, 39, "M", "U", base_time - timedelta(hours=1)), # Simulated Update
    ("PT-671", "Alice Wong",  55.6, 158, 25, "F", "I", base_time)
]

# 3. Create DataFrame
df_bronze_input = spark.createDataFrame(data, schema=schema)

# 4. Write to Bronze Table (Delta is recommended for CDC)
# If you are using Databricks or Delta Lake:
df_bronze_input.write.format("delta").mode("append").saveAsTable("patient_data.bronze_patients.bronze_patients_cdc")

df_bronze_input.show()